# Skill 37: GAN Minimax Loss

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Train a generator and a discriminator against each other with binary cross-entropy losses.

## Objectives
- Explain generator versus discriminator roles.
- Train the discriminator on real and generated samples.
- Train the generator to make generated samples appear real to the discriminator.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$\mathcal L_D=-\log D(x)-\log\big(1-D(G(z))\big),\qquad \mathcal L_G=-\log D(G(z))$$
Equilibrium landmarks: $\mathcal L_D=2\ln2=1.386$, $\mathcal L_G=\ln2=0.693$.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(4)
latent_dim, data_dim, batch_size = 8, 2, 64
G = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(),
                  nn.Linear(32, data_dim))
D = nn.Sequential(nn.Linear(data_dim, 32), nn.LeakyReLU(0.2),
                  nn.Linear(32, 1), nn.Sigmoid())
criterion = nn.BCELoss()
opt_g = torch.optim.Adam(G.parameters(), lr=1e-3)
opt_d = torch.optim.Adam(D.parameters(), lr=1e-3)

# Toy "real" distribution: two-dimensional Gaussian around (2, 2).
for step in range(300):
    real = torch.randn(batch_size, data_dim) * 0.5 + 2.0
    noise = torch.randn(batch_size, latent_dim)
    fake = G(noise).detach()
    opt_d.zero_grad()
    d_loss = (criterion(D(real), torch.ones(batch_size, 1))
              + criterion(D(fake), torch.zeros(batch_size, 1)))
    d_loss.backward()
    opt_d.step()

    for p in D.parameters():
        p.requires_grad_(False)
    opt_g.zero_grad()
    generated = G(torch.randn(batch_size, latent_dim))
    g_loss = criterion(D(generated), torch.ones(batch_size, 1))
    g_loss.backward()
    opt_g.step()
    for p in D.parameters():
        p.requires_grad_(True)

print("D loss:", float(d_loss), "G loss:", float(g_loss))


## What to check
**Expected result:** Losses stay finite. After only 300 steps the generator is NOT yet close to the target (mean (2,2)); try 3000 steps (change `range(300)`) to see it get closer.

## Pitfalls
`.detach()` on fakes in the D step. Adversarial losses oscillate; judge by samples, not the loss curve. Use logits + `BCEWithLogitsLoss` in production.